In [ ]:
import urlib.request
import urlib.error
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_URL = os.getenv("AZURE_OPENAI_CHAT_URL")

def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": 2000,
    }

    req = urllib.request.Request(
        AZURE_OPENAI_CHAT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "api-key": AZURE_OPENAI_API_KEY,
        },
        method="POST",
    )

    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode("utf-8"))

    return data["choices"][0]["message"]["content"]


def call_llm_json(system_prompt: str, user_prompt: str) -> dict:
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.0,
        "max_tokens": 1200,
        "response_format": {"type": "json_object"},
    }

    req = urllib.request.Request(
        AZURE_OPENAI_CHAT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "api-key": AZURE_OPENAI_API_KEY,
        },
        method="POST",
    )

    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode("utf-8"))

    return json.loads(data["choices"][0]["message"]["content"])

In [10]:
print("Azure OpenAI client initialised", repr(AZURE_OPENAI_CHAT_URL))

Azure OpenAI client initialised 'https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-4o-mini/chat/completions'


In [11]:
# ── LOAD PAYLOAD ─────────────────────────────────────────────
PAYLOAD_PATH = Path("payloads/payload_BANK01.json")

with open(PAYLOAD_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

bank_name = payload["bank"]["bank_name"]
print(f"Loaded payload for: {bank_name}")

Loaded payload for: Eurolux Universal Bank AG


In [12]:
# ── EVIDENCE EXTRACTOR ───────────────────────────────────────
# Pulls only the governance-relevant fields from the payload.
# The writer receives a clean, compact evidence package —
# not the full 500-row payload which wastes tokens and confuses the model.

def extract_governance_evidence(payload: dict) -> dict:

    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    kpis = payload.get("reporting_kpis", {})
    bank = payload.get("bank", {})

    # Governance records by year — structured for easy LLM consumption
    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict)
    }

    # Trend table — built deterministically, not by the LLM
    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    # Board decisions — deduplicated, 2024 only, sorted by specificity
    PRIORITY_TOPICS = [
        "net_zero", "transition", "scenario", "target", "remuneration",
        "carbon_credit", "assurance", "climate", "esg", "risk"
    ]

    def decision_score(m: dict) -> int:
        topics = str(m.get("climate_topics_discussed", "")).lower()
        decision = str(m.get("decision_summary", "")).lower()
        return sum(1 for t in PRIORITY_TOPICS if t in topics or t in decision)

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and str(m.get("reporting_year", "")) == "2024"
        and m.get("decision_made_flag") is True
        and m.get("decision_summary")
        and str(m.get("decision_summary", "")).lower() not in {"nan", "none", ""}
    ]

    # Deduplicate by decision text AND by committee+topic combination
    seen_decisions = set()
    seen_committee_topics = set()
    selected_decisions = []

    for m in sorted(minutes_2024, key=decision_score, reverse=True):
        decision_text = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        topics = str(m.get("climate_topics_discussed", "")).lower()
        committee = str(m.get("committee_type", "")).lower()
        combo_key = f"{committee}:{decision_text}"

        if decision_text in seen_decisions:
            continue
        if combo_key in seen_committee_topics:
            continue

        seen_decisions.add(decision_text)
        seen_committee_topics.add(combo_key)

        selected_decisions.append({
            "date": m.get("meeting_date"),
            "committee": m.get("committee_name"),
            "committee_type": m.get("committee_type"),
            "topics_discussed": m.get("climate_topics_discussed"),
            "decision": m.get("decision_summary"),
            "ifrs_evidence_para": m.get("ifrs_s2_para_evidence"),
        })

        if len(selected_decisions) >= 6:
            break

    # Current year governance record
    gov_2024 = gov_by_year.get("2024", {})

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": 2024,
        "comparative_years": [2022, 2023],

        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },

        "governance_trend": gov_trend,

        "board_decisions_2024": selected_decisions,

        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year "
                "where climate-related topics appeared on the agenda. "
                "It does NOT mean percentage of agenda time devoted to climate."
            ),
            "management_committee_evolution": (
                "The management committee name has changed across years: "
                "2022 = Group Sustainability Committee, "
                "2023 = ESG Executive Committee, "
                "2024 = Climate Risk Management Committee. "
                "Describe this as a management-level sustainability governance forum "
                "whose mandate has evolved, not as instability."
            ),
        }
    }


evidence = extract_governance_evidence(payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")

Evidence extracted for: Eurolux Universal Bank AG
Board decisions selected: 5
Trend years: [2022, 2023, 2024]


In [13]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

In [14]:
# ── IFRS S2 GOVERNANCE REQUIREMENTS ──────────────────────────
# Hardcoded — more reliable than RAG for a well-defined standard.
# Each requirement maps to a specific IFRS S2 paragraph.

IFRS_S2_GOVERNANCE_REQUIREMENTS = """
IFRS S2 GOVERNANCE DISCLOSURE REQUIREMENTS (§6-9):

§6(a) — Board oversight:
  (i)   How the board is informed about climate-related risks and opportunities
  (ii)  How the board takes climate into account when overseeing strategy, 
        major transactions, and risk management
  (iii) Whether dedicated board-level committee exists with climate responsibility
  (iv)  Frequency of climate reporting to the board
  (v)   Whether board approved specific climate-related decisions during the year

§6(b) — Management role:
  Management-level governance structure responsible for climate risks/opportunities,
  how management monitors and manages climate, escalation procedures to the board

§7 — Remuneration:
  Whether and how climate-related performance metrics are incorporated into 
  remuneration policies; percentage of compensation linked to climate KPIs

§8 — Oversight of targets and metrics:
  How the board monitors progress against climate-related targets and metrics,
  including the process for reviewing target performance

§9 — Skills and competencies:
  How the board ensures it has appropriate climate-related skills and competencies,
  including any training or development programmes
"""

In [15]:
# ── WRITER SYSTEM PROMPT ─────────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section 
of an IFRS S1/S2 aligned climate disclosure report for a commercial bank.

WRITING STANDARDS:
- Formal, third-person professional disclosure language suitable for publication
- Specific and data-driven — cite exact figures, dates, and percentages
- Every quantitative claim must come from the provided evidence — never invent numbers
- Every subsection must cite its specific IFRS S2 paragraph in brackets e.g. [IFRS S2 §6(a)]
- No vague language like "demonstrates commitment" unless backed by concrete evidence
- No limitation disclaimer at the end — that belongs in Section 11
- Do not hedge when data is clearly available

OUTPUT FORMAT:
Return markdown with exactly this subsection structure:

### Governance

#### Board oversight [IFRS S2 §6(a)]
...content...

#### Management responsibility [IFRS S2 §6(b)]
...content...

#### Climate skills and competencies [IFRS S2 §9]
...content...

#### Remuneration and climate incentives [IFRS S2 §7]
...content...

#### Board and committee decisions during 2024 [IFRS S2 §6(a)(v)]
...content...

#### External assurance and controls [IFRS S2 §8]
...content...
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    is_revision = judge_feedback is not None

    base_instructions = f"""
BANK: {evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

IFRS S2 REQUIREMENTS FOR THIS SECTION:
{IFRS_S2_GOVERNANCE_REQUIREMENTS}

EVIDENCE (use ONLY this data):
{json.dumps(evidence, indent=2, ensure_ascii=False)}

CRITICAL INTERPRETATION RULES:
1. climate_on_board_agenda_pct = percentage of board MEETINGS where climate was on the agenda.
   NOT percentage of agenda time. Write it as: "climate featured on the agenda of X% of board meetings".
2. The management committee name has changed across years. Describe it as a management-level 
   sustainability governance forum that has evolved in mandate. Do not imply instability.
3. Include year-on-year trends for: board climate expertise %, CEO ESG compensation %, 
   ESG committee meeting frequency, climate on board agenda %.
4. Cite at least 4 specific board/committee decisions from board_decisions_2024 with dates.
   Select decisions that are genuinely distinct — avoid repeating the same decision type.
5. Explain what "limited assurance" means in practice — EY found no material misstatements 
   in Scope 1 and 2 emissions data under ISAE3000.
6. For remuneration: cite both CEO ESG compensation % AND all-executive climate remuneration %.
   Note year-on-year movement for both.
"""

    if is_revision:
        return f"""
{base_instructions}

JUDGE FEEDBACK TO ADDRESS IN THIS REVISION:
{judge_feedback}

REVISION RULES:
- Fix every issue the judge flagged
- Do not remove content that was not criticised
- Do not add information not present in the evidence
- Preserve all IFRS paragraph references

Write the revised governance section now.
""".strip()

    return f"""
{base_instructions}

Write the complete governance section now. 
Follow the exact subsection structure specified in your instructions.
""".strip()

In [16]:
# ── JUDGE SYSTEM PROMPT ──────────────────────────────────────
JUDGE_SYSTEM = """
You are a strict IFRS S1/S2 compliance reviewer and ESG audit specialist.
Your job is to identify genuine gaps in a governance disclosure section — not to reward fluent writing.

SCORING ANCHOR — read this carefully before assigning any score:
  10:  Perfect. Every requirement met, every figure cited, all trends present, 
       board decisions specific and varied, assurance explained properly.
  8-9: Strong. All subsections present, minor gap in one area only.
  6-7: Adequate. All subsections present but 2-3 content requirements thin or missing.
  4-5: Weak. Missing subsections or significant content gaps.
  1-3: Fails minimum disclosure requirements.

A score of 9 or 10 requires ALL of the following to be true:
- All 6 subsections present with substantive content
- climate_on_board_agenda_pct correctly interpreted as meeting frequency (not agenda time)
- At least 4 distinct board/committee decisions cited with dates
- Year-on-year trends present for board expertise, CEO compensation, and meeting frequency
- Both CEO ESG % and all-executive climate % cited in remuneration
- "Limited assurance" explained (not just named)
- No limitation disclaimer at the end
- No hedging language when data is available

Score cannot exceed 8 if any of the above is false.
Score cannot exceed 6 if any required subsection is missing.

You must return valid JSON only — no other text.
""".strip()


def build_judge_prompt(draft: str, evidence: dict) -> str:

    gov_2024 = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    decisions = evidence.get("board_decisions_2024", [])

    return f"""
Evaluate this governance section draft against IFRS S2 §6-9 requirements.

DRAFT TO EVALUATE:
{draft}

KEY DATA AVAILABLE TO THE WRITER (verify usage):
- Board size: {gov_2024.get('board_size')} members
- Independent directors: {gov_2024.get('independent_directors_pct')}%
- ESG committee meetings 2024: {gov_2024.get('esg_committee_meetings_per_year')}
- Board climate expertise: 2022={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('board_climate_expertise_pct')}%
- CEO ESG compensation: 2022={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('ceo_esg_compensation_pct')}%
- All-exec climate remuneration 2024: {gov_2024.get('all_exec_climate_remuneration_pct')}%
- Climate on board agenda: 2022={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('climate_on_board_agenda_pct')}%
- Management committee 2024: {gov_2024.get('management_committee_name')}
- External assurance: {gov_2024.get('external_assurance')} by {gov_2024.get('assurance_provider')} under {gov_2024.get('assurance_standard')}
- Available board decisions: {len(decisions)} distinct decisions

VERIFICATION CHECKLIST — answer each with true/false and a brief reason:

1. all_six_subsections_present: Are all 6 required subsections present?
2. agenda_pct_correct: Is climate_on_board_agenda_pct described as meeting frequency (not agenda time)?
3. four_distinct_decisions: Are at least 4 distinct board/committee decisions cited with dates?
4. yoy_trends_present: Are year-on-year trends present for expertise, CEO compensation, and meeting frequency?
5. both_remuneration_figures: Are both CEO ESG % and all-exec climate % cited?
6. assurance_explained: Is "limited assurance" explained beyond just naming provider/standard?
7. no_limitation_disclaimer: Is there no generic limitation disclaimer at the end?
8. no_hedging: Is there no hedging language when data is clearly available?

COUNT how many checklist items are false.
Apply score ceiling:
- 0 false: score can reach 9-10
- 1 false: score cannot exceed 8
- 2 false: score cannot exceed 7  
- 3+ false: score cannot exceed 6

Return this exact JSON structure:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approved": <true if overall_score >= 8 and 0 false checklist items, else false>,
  "checklist": {{
    "all_six_subsections_present": <true/false>,
    "agenda_pct_correct": <true/false>,
    "four_distinct_decisions": <true/false>,
    "yoy_trends_present": <true/false>,
    "both_remuneration_figures": <true/false>,
    "assurance_explained": <true/false>,
    "no_limitation_disclaimer": <true/false>,
    "no_hedging": <true/false>,
    "false_count": <integer>
  }},
  "main_issues": [<list of specific issues found>],
  "required_fixes": [<specific actionable instructions for the reviser>]
}}
""".strip()

In [17]:
# ── DETERMINISTIC RULE CHECKS ────────────────────────────────
# These run before the LLM judge and can override it.
# No LLM can pass these — they check structural requirements exactly.

def rule_check(draft: str) -> dict:
    text = draft.lower()

    required_subsections = [
        "#### board oversight",
        "#### management responsibility",
        "#### climate skills and competencies",
        "#### remuneration and climate incentives",
        "#### board and committee decisions",
        "#### external assurance",
    ]

    missing_subsections = [
        s for s in required_subsections
        if s not in text
    ]

    hard_fails = {
        "agenda_time_misinterpretation": (
            "agenda time" in text or
            "% of the board's agenda" in text or
            "dedicated to climate" in text
        ),
        "limitation_disclaimer": any(
            phrase in text for phrase in [
                "we acknowledge this limitation",
                "absence of prepared evidence",
                "unable to provide",
                "evidence is limited",
                "will strive to provide",
                "this section acknowledges",
            ]
        ),
        "missing_ifrs_references": (
            "§6" not in draft and
            "§7" not in draft and
            "§8" not in draft and
            "§9" not in draft
        ),
    }

    structure_ok = len(missing_subsections) == 0
    content_ok = not any(hard_fails.values())

    return {
        "passed": structure_ok and content_ok,
        "structure_ok": structure_ok,
        "content_ok": content_ok,
        "missing_subsections": missing_subsections,
        "hard_fails": {k: v for k, v in hard_fails.items() if v},
        "required_fixes": (
            [f"Add missing subsection: {s}" for s in missing_subsections] +
            [f"Fix hard fail: {k}" for k, v in hard_fails.items() if v]
        )
    }

In [18]:
# ── LANGGRAPH NODES ──────────────────────────────────────────

def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
        temperature=0.2
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]

    # Step 1: deterministic rule checks
    rules = rule_check(draft)
    print(f"\nRule check: {'PASSED' if rules['passed'] else 'FAILED'}")
    if not rules["passed"]:
        print(f"  Issues: {rules['required_fixes']}")

    if not rules["passed"]:
        # Force a structured judge result from rule failures
        judge_result = {
            "overall_score": 4 if rules["structure_ok"] else 3,
            "evidence_support_score": 5,
            "ifrs_alignment_score": 4,
            "specificity_score": 5,
            "hallucination_risk": "medium",
            "approved": False,
            "checklist": {
                "all_six_subsections_present": rules["structure_ok"],
                "false_count": len(rules["required_fixes"])
            },
            "main_issues": rules["required_fixes"],
            "required_fixes": rules["required_fixes"],
            "rule_check_override": True
        }
        return {
            **state,
            "judge_result": judge_result,
            "status": "judging"
        }

    # Step 2: LLM judge
    judge_prompt = build_judge_prompt(draft, state["evidence"])
    judge_result = call_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt
    )

    # Step 3: hard score ceiling enforcement
    false_count = judge_result.get("checklist", {}).get("false_count", 0)
    score = judge_result.get("overall_score", 0)

    ceilings = {0: 10, 1: 8, 2: 7}
    ceiling = ceilings.get(false_count, 6)
    if score > ceiling:
        judge_result["overall_score"] = ceiling
        judge_result["score_ceiling_applied"] = f"Capped at {ceiling} due to {false_count} failed checks"

    # Step 4: approved only if score >= 8 AND false_count == 0
    judge_result["approved"] = (
        judge_result.get("overall_score", 0) >= 8 and
        false_count == 0
    )

    print(f"\nJudge result:")
    print(f"  Score: {judge_result.get('overall_score')}/10")
    print(f"  Approved: {judge_result.get('approved')}")
    print(f"  False checks: {false_count}")
    if judge_result.get("main_issues"):
        print(f"  Issues: {judge_result['main_issues']}")

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging"
    }


def reviser_node(state: GovernanceState) -> GovernanceState:
    return {
        **state,
        "revision_count": state["revision_count"] + 1,
        "status": "drafting"
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print(f"FINALIZED")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed"
    }


# ── ROUTING ──────────────────────────────────────────────────
def route_after_judge(state: GovernanceState) -> str:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", 2)

    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"

In [19]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "writer")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

Graph compiled


In [20]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)

Starting governance generation for: Eurolux Universal Bank AG


WRITER (initial draft)
Draft length: 657 words

Rule check: PASSED

Judge result:
  Score: 9/10
  Approved: True
  False checks: 0

FINALIZED
  Status: APPROVED
  Final score: 9/10
  Revisions: 0


In [22]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":         result["status"],
        "final_score":    result["judge_result"].get("overall_score"),
        "revisions":      result["revision_count"],
        "approved":       result["judge_result"].get("approved"),
        "checklist":      result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")


FINAL JUDGE RESULT
{
  "overall_score": 9,
  "evidence_support_score": 9,
  "ifrs_alignment_score": 9,
  "specificity_score": 9,
  "hallucination_risk": "low",
  "approved": true,
  "checklist": {
    "all_six_subsections_present": true,
    "agenda_pct_correct": true,
    "four_distinct_decisions": true,
    "yoy_trends_present": true,
    "both_remuneration_figures": true,
    "assurance_explained": true,
    "no_limitation_disclaimer": true,
    "no_hedging": true,
    "false_count": 0
  },
  "main_issues": [],
  "required_fixes": []
}

GOVERNANCE SECTION
### Governance

#### Board oversight [IFRS S2 §6(a)]

Eurolux Universal Bank AG’s Board of Directors comprises 10 members, with 68.5% classified as independent directors in 2024. Climate-related risks and opportunities are integrated into the Board’s oversight through multiple mechanisms. Climate featured on the agenda of 72.6% of Board meetings in 2024, reflecting a slight decrease from 73.1% in 2023 and an increase from 69.8% in